# ¿Cómo sabes que tu RAG funciona? Métricas de evaluación desde cero


Construir un RAG es la parte fácil; saber si responde bien es el problema real. En esta charla implementamos desde cero, en Python puro, las métricas que usan frameworks como RAGAS y DeepEval: faithfulness, contextual precision y contextual recall. Primero el pseudocódigo y la intuición matemática, luego el mismo cálculo usando las librerías, para que quede claro qué hace cada una por debajo. Cerramos con las trampas prácticas: por qué un juez LLM da scores inconsistentes, cómo se calibran, y cuánto cuesta evaluar a volumen real.

---

## Table of Contents

1. Intro
2. ¿Cómo funciona un LLM como Juez?
3. ¿Cómo puntúa un LLM como Juez?
4. Limitaciones y Riesgos de un LLM como Juez
5. Faithfulness RAGAS
6. Faithfulness DEEPEVAL
7. Implementación
8. Implementación DeepEval
9. Calibración

---
## Intro

Retrieval Augmented Generation (RAG) es una arquitectura o patrón de diseño que permite conectar un modelo de lenguage (LLM) con una fuente de datos externa para mejorar sus respuestas.

```mermaid
flowchart LR
    Input -- pregunta --> Retriever
    Retriever -- similitudes --> VectorStore
    VectorStore -- documentos --> Retriever
    Retriever -- pregunta y documentos --> LLM
    LLM -- respuesta --> Output
```

---

## ¿Cómo funciona un LLM-as-a-Judge?

A primera vista parece sencillo el funcionamiento de un LLM como Juez, un modelo evalua la salida de otro modelo sin embargo hay que considerar ciertos aspectos críticos para determinar si el resultado que se produce es confiable y reproducible.

Para la evaluación de las respuestas de un modelo se refina el prompt con el que se evaluara la respuesta recuperando el contexto y una rúbrica de evaluación que se injectarán en el prompt a enviar al juez y la salida debe ser apto para ser leído por la máquina es decir un formato json.

Para que la salida sea consistente se debe calibrar usando técnicas de aprendizaje automático.

```mermaid
flowchart TD
    input["`Salida del Modelo Evaluado`"]
    evaluacion["`Capa de Evaluación`"]
    contexto["`Contexto Recuperado`"]
    rubrica["`Rubrica de Evaluación`"]
    juez["`LLM como Juez`"]
    salida["`Salida Estructurada`"]
    calibracion["`Capa de Calibración`"]
    revision["`Revisión Humana`"]
    resultado["`Resultado de la Evaluación`"]


    input --> evaluacion
    contexto -.-> evaluacion
    rubrica -.-> evaluacion
    evaluacion --> juez
    juez --> salida
    salida -.-> calibracion
    salida -.-> revision
    revision -.-> calibracion
    calibracion --> resultado
```

---
## ¿Cómo puntúa un LLM-as-a-Judge?

Existen 3 tipos de tipos de puntuaciones.

1. Pairwise Prompt

El Juez se le presenta en el prompt dos respuestas y después se le pide que seleccione la mejor.

2. Pointwise Prompt

El Juez se le presenta en el prompt la pregunta y respuesta, después se le pide que califique usando una escala tipo 1-5.

3. Binarywise Prompt

El Juez se le presenta en el prompt un enunciado y se le pide que distinga entre verdadero o falso dependiendo de lo que se evalúe

---

## Limitaciones y riesgos de un LLM como Juez

Utilizar un LLM como Juez puede traer beneficios, sin embargo, puede introducir puntos ciegos en el proceso de evaluación y entre las principales dificultades se encuentran:

### Sesgo de posición

El juez puede dar preferencia a una propuesta que se encuentre al último o al principio del prompt dependiendo del modelo.

### Sesgo de verbosidad

El juez puede dar preferencia a respuestas más largas o dar una puntuación a más alta a respuestas largas y prolijas en lugar de a una breve y clara.

### Sesgo de auto-valorización

Es posible que un modelo de preferencia a respuestas redactadas por su propia familia de modelos, es decir, al modelo evaluador le suele gustarle la redacción y estructura que le resultan familiares.

### Sensibilidad a las indicaciones

Hay que tener mucho cuidado porque un cambio en la rúbrica puede afectar a las puntuaciones finales.

### Desviación del Modelo

Tomar en cuenta que los proveedores de modelos actualizan de forma constante y sin previo aviso. Cuando ocurre esto se puede desajustar las calificaciones de los modelos.

### Optimización Adversarial

Al ajustar el modelo con el mismo evaluador en el entrenamiento puede resultar que el modelo aprenda los patrones que ofrecen mayor puntuación en lugar ofrecer mejores respuestas.

---

## Faithfulness

Es una métrica que mide como la consistencia factual de una respuesta esta con el contexto recuperado, es decir, mide que tanto de la respuesta esta fundamentado en el contexto recuperado y si la respuesta empieza a desviarse del contexto la métrica penaliza la respuesta dandole un puntaje menor. El rango de la métrica es entre 0 y 1.

Sin embargo dependiendo de la librería que se utilice es como se evalúa la respuesta.

---

## Faithfulness RAGAS

```
INPUT:
  q   : question           (string)
  a   : answer/output      (string)
  c   : context = passages (list[string])
  LLM : judge model
OUTPUT:
  F   : faithfulness score in [0, 1]

FUNCTION ragas_faithfulness(q, a, c, LLM):

  # Step 1 — Statement extraction FROM THE ANSWER
  S = LLM.extract_statements(q, a)

  # Step 2 — Verify each statement AGAINST THE CONTEXT
  supported = 0
  FOR s_i IN S:
      # "can s_i be inferred from context?" -> Yes/No (+reason)
      verdict = LLM.verify(s_i, c)
      # <-- POSITIVE support required
      IF verdict == YES:                
          supported += 1

  # Step 3 — Score
  F = supported / len(S)
  RETURN F
```

---

## Faithfulness DeepEval

```
INPUT:
  input             : user query (string)     # required param, NOT scored
  actual_output     : generated output (string)
  retrieval_context : passages (list[string])
  LLM               : judge model
OUTPUT:
  score  : faithfulness score in [0, 1]

FUNCTION deepeval_faithfulness(actual_output, retrieval_context, LLM):

  # Step 1 — Extract TRUTHS from the CONTEXT
  truths = LLM.generate_truths(retrieval_context, limit)

  # Step 2 — Extract CLAIMS from the OUTPUT
  claims = LLM.generate_claims(actual_output)

  # Step 3 — Per-claim verdict: does it CONTRADICT the truths?
  verdicts = []                          # each verdict in {"yes", "no", "idk"}
  FOR claim IN claims:
      v = LLM.generate_verdict(claim, truths)
          # "no"  ONLY if truths DIRECTLY contradict the claim
          # "idk" if not mentioned / unverifiable
          # "yes" if it agrees
      verdicts.append(v)

  # Step 4 — Score = fraction of claims NOT contradicted
  faithful = COUNT(v IN verdicts WHERE v != "no")   # <-- "yes" AND "idk" both pass
  score = faithful / len(verdicts)

  RETURN score
```

## Implementación Faithfulness RAGAS

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from syntok import segmenter


def split_sentences(text: str) -> list[str]:
    """Segment a text blob into sentences with syntok."""
    sentences: list[str] = []
    for paragraph in segmenter.analyze(text):
        for sentence in paragraph:
            rebuilt = "".join(token.spacing + token.value for token in sentence).strip()
            if rebuilt:
                sentences.append(rebuilt)
    return sentences

In [3]:
from pydantic import BaseModel


class RagasEntailment(BaseModel):
    """One claim's entailment verdict plus the judge's self-reported confidence."""

    entailed: bool = False
    justification: str = ""

In [4]:
DEFAULT_SYSTEM_PROMPT = (
    "Eres un evaluador experto de conversaciones entre usuarios y asistentes de "
    "IA. Evalúas un único turno según una rúbrica y devuelves una puntuación "
    "numérica junto con una justificación breve. Sé objetivo y responde siempre "
    "en español."
)

In [5]:
VERIFY_RAGAS = """\
¿Puede inferirse la siguiente afirmación a partir del contexto recuperado?
Devuelve:
- entailed: true si la afirmación se deduce del contexto, false si no se deduce o lo \
contradice.
- justification: una justificación breve en español.
Afirmación: {claim}"""

In [6]:
from openai import OpenAI

client = OpenAI()


def call_openai(messages: list[dict], schema: type[BaseModel]) -> BaseModel:
    """Wrapper alrededor de Structured Outputs: devuelve el modelo pydantic ya validado."""
    completion = client.chat.completions.parse(
        model="gpt-4o-mini",
        messages=messages,  # type: ignore[arg-type]
        response_format=schema,
    )
    parsed = completion.choices[0].message.parsed
    if parsed is None:
        raise ValueError("El modelo no devolvió una salida estructurada válida.")
    return parsed

In [7]:
def faithfulness_ragas(prompt: str, response: str, context: str):
    claims = split_sentences(response)

    supported = 0
    for claim in claims:
        content = VERIFY_RAGAS.format(**{"claim": claim})

        result = call_openai(
            messages=[
                {"role": "system", "content": DEFAULT_SYSTEM_PROMPT},
                {
                    "role": "user",
                    "content": """Pregunta: {prompt}\n\nConexto: {context}\n\nContenido: {content}""".format(
                        **{"prompt": prompt, "context": context, "content": content}
                    ),
                },
            ],
            schema=RagasEntailment,
        )

        print(result)

        supported += int(result.entailed)  # type: ignore

    return supported / len(claims)

In [8]:
faithfulness_ragas(
    "Quien fue Nikola Tesla? y Cuáles fueron sus contribuciones?",
    "Fue un físico del sigo XIX y contribuyo al diseño de la corriente alterna",
    "Nikola Tesla fue un ingeniero, futurista e inventor serbio-estadounidense. Es conocido por sus contribuciones al diseño del sistema moderno de suministro eléctrico de corriente alterna. "
    "Nacido y criado en el Imperio austrohúngaro, Tesla estudió ingeniería y física en la década de 1870, aunque no obtuvo ningún título.",
)

entailed=True justification='El contexto menciona que Nikola Tesla fue un ingeniero y inventor conocido por sus contribuciones al suministro eléctrico de corriente alterna y que estudió ingeniería y física en la década de 1870, lo que permite inferir que fue un físico del siglo XIX y que efectivamente contribuyó al diseño de la corriente alterna.'


1.0

In [9]:
faithfulness_ragas(
    "Que dia es hoy?",
    "Hoy es Lunes 13 de Agosto de 2026",
    "La fecha de hoy es 11 de Agosto de 2026",
)

entailed=False justification='El contexto indica que la fecha es 11 de Agosto de 2026, por lo tanto, no puede afirmarse que hoy sea Lunes 13 de Agosto de 2026.'


0.0

## Implementación Faithfulness DeepEval

In [10]:
GENERATE_TRUTHS = """\
Extrae las verdades o hechos presentes en el contexto recuperado. Cada verdad debe ser un \
enunciado atómico y verificable tomado únicamente del contexto. Devuelve también un breve \
resumen en español.
Contexto: {context}"""

In [11]:
VERIFY_DEEPEVAL = """\
¿Las siguientes verdades contradicen la afirmación? Asigna 0 SOLO si las verdades contradicen \
directamente la afirmación. Asigna 1 si la afirmación concuerda con las verdades o si no se \
menciona (no verificable). Justifica brevemente en español.
Verdades:
{truths}
Afirmación: {claim}"""

In [12]:
class Truths(BaseModel):
    """Ground-truth facts extracted from the retrieved context."""

    truths: list[str] = []
    summary: str = ""


class ScoreResponse(BaseModel):
    """The structured result a judge requests from the model for a rubric score."""

    score: float
    justification: str

In [13]:
def faithfulness_deepeval(prompt: str, response: str, context: str):

    truths = call_openai(
        messages=[
            {"role": "system", "content": DEFAULT_SYSTEM_PROMPT},
            {
                "role": "user",
                "content": GENERATE_TRUTHS.format(**{"context": context}),
            },
        ],
        schema=Truths,
    )

    claims = split_sentences(response)

    veredicts = []
    for claim in claims:
        veredict = call_openai(
            messages=[
                {"role": "system", "content": DEFAULT_SYSTEM_PROMPT},
                {
                    "role": "user",
                    "content": VERIFY_DEEPEVAL.format(
                        **{"truths": truths.truths, "claim": claim}  # type: ignore
                    ),
                },
            ],
            schema=ScoreResponse,
        )

        veredicts.append(veredict)

    print(veredicts)

    return sum([v.score for v in veredicts]) / len(veredicts)

In [14]:
faithfulness_deepeval(
    "Quien fue Nikola Tesla? y Cuáles fueron sus contribuciones?",
    "Fue un físico del sigo XIX y contribuyo al diseño de la corriente alterna",
    "Nikola Tesla fue un ingeniero, futurista e inventor serbio-estadounidense. Es conocido por sus contribuciones al diseño del sistema moderno de suministro eléctrico de corriente alterna. "
    "Nacido y criado en el Imperio austrohúngaro, Tesla estudió ingeniería y física en la década de 1870, aunque no obtuvo ningún título.",
)

[ScoreResponse(score=1.0, justification='Las verdades presentadas son consistentes con la afirmación. Nikola Tesla fue un ingeniero e inventor del siglo XIX y su contribución al diseño del sistema de corriente alterna es un hecho conocido. No hay ninguna verdad que contradiga directamente la afirmación.')]


1.0

In [15]:
faithfulness_deepeval(
    "Que dia es hoy?",
    "Hoy es Lunes 13 de Agosto de 2026",
    "La fecha de hoy es 11 de Agosto de 2026",
)

[ScoreResponse(score=0.0, justification='La afirmación de que hoy es Lunes 13 de Agosto de 2026 contradice la verdad de que la fecha de hoy es 11 de Agosto de 2026, ya que no puede ser ambas fechas al mismo tiempo.')]


0.0

---

## Implementación DeepEval

In [16]:
import os
import pathlib

from uuid import uuid4

from dotenv import load_dotenv

from typing import TypedDict

from pydantic import SecretStr, Field, BaseModel

from langchain.agents import create_agent
from langgraph.graph import StateGraph
from langchain.agents.middleware import dynamic_prompt
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import MarkdownHeaderTextSplitter

from deepeval.integrations.langchain import CallbackHandler
from deepeval.dataset import EvaluationDataset, Golden
from deepeval.test_case import LLMTestCase
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from deepeval.tracing import observe, update_current_span

load_dotenv()

True

---

## Vector Store

In [17]:
md_text_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("#", "header1"),
        ("##", "header2"),
        ("###", "header3"),
    ],
    strip_headers=False,
)

path = pathlib.Path("../data/docs_rag")

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    check_embedding_ctx_length=False,
    api_key=SecretStr(os.environ["OPENAI_API_KEY"]),
)

vector_store = Chroma(
    collection_name="qbr",
    embedding_function=embeddings,
    persist_directory="../.chroma_langchain_db",
)

for file in path.glob("*.md"):
    with open(file) as blob:
        doc = blob.read()
    documents = md_text_splitter.split_text(doc)

    vector_store.add_documents(
        documents=documents,
        ids=[str(uuid4()) for _ in range(len(documents))],
    )

retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.2, "k": 50},
)

---

## Retrieval Evaluator

In [18]:
class RetrievalEvaluator(BaseModel):
    """Classify retrieved documents based on how relevant it is to the user's question."""

    binary_evaluation: str = Field(
        description="Documents are relevant to the question, 'yes' or 'no'"
    )


retrieval_evaluator = create_agent(
    model="gpt-5.4-nano-2026-03-17",
    response_format=RetrievalEvaluator,
    system_prompt=(
        "Eres un evaluador de recuperación de documentos responsable de determinar si un documento recuperado es relevante para la pregunta del usuario. "
        "Considera que un documento es relevante si contiene palabras clave, conceptos, información relacionada o significado semántico que contribuya a responder la pregunta del usuario. "
        "Emite únicamente una puntuación binaria: "
        '"yes": si el documento es relevante para la pregunta.'
        '"no": si el documento no es relevante para la pregunta.'
        'No agregues explicaciones ni información adicional. La salida debe ser exclusivamente "yes" o "no".'
    ),
)

---
## Question Rewritter

In [19]:
class Question(BaseModel):
    """Question rewritten to give more context to the RAG or web Search"""

    question: str = Field(
        description="question rewritten to give more context to the LLM"
    )


question_rewriter = create_agent(
    model="gpt-5.4-nano-2026-03-17",
    response_format=Question,
    system_prompt=(
        "Eres un reescritor de preguntas encargado de transformar una pregunta de entrada en una versión mejorada y optimizada para búsquedas mediante sistemas RAG (Retrieval-Augmented Generation). "
        "Analiza la pregunta de entrada e identifica su intención semántica subyacente, es decir, qué información busca realmente obtener el usuario. "
        "Devuelve únicamente la pregunta reescrita, sin explicaciones ni comentarios adicionales. "
    ),
)

---
## Main Agent

In [20]:
@dynamic_prompt
def build_prompt(request) -> str:
    documents = request.runtime.context["documents"]
    context_text = "\n\n".join(d.page_content for d in documents)
    return (
        "Eres un asistente especializado en responder preguntas utilizando información recuperada de un sistema RAG. "
        "Utiliza exclusivamente el contexto proporcionado para responder la pregunta del usuario. Si la información necesaria para responder no está disponible o no puedes determinar la respuesta con suficiente certeza a partir del contexto, indica que no lo sabes. "
        "Responde de forma clara y concisa, con un máximo de tres oraciones. "
        f"Contexto recuperado: {context_text}"
    )


main_agent = create_agent(
    model="gpt-5.4-nano-2026-03-17",
    middleware=[build_prompt],
)

---
## LangGraph

In [21]:
def regenerate_query(state):
    question = state["question"]

    better_question = question_rewriter.invoke(
        {"messages": [{"role": "user", "content": question}]}
    )

    return {
        "question": better_question["structured_response"].question,
    }


@observe(
    metrics=[
        AnswerRelevancyMetric(verbose_mode=True),
        FaithfulnessMetric(verbose_mode=True),
    ]
)
def answer_question(state):
    question = state["question"]
    documents = retriever.invoke(question)

    answer = main_agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        context={"documents": documents},  # type: ignore
    )

    update_current_span(
        test_case=LLMTestCase(
            input=question,
            actual_output=answer["messages"][-1].content,
            retrieval_context=[d.page_content for d in documents],
        )
    )

    return {
        "question": question,
        "answer": answer["messages"][-1].content,
    }

In [22]:
class RagState(TypedDict):
    question: str
    answer: str


workflow = StateGraph(RagState)
workflow.add_node("regenerate_query_node", regenerate_query)
workflow.add_node("answer_question_node", answer_question)

workflow.add_edge("__start__", "regenerate_query_node")
workflow.add_edge("regenerate_query_node", "answer_question_node")
workflow.add_edge("answer_question_node", "__end__")


app = workflow.compile()

In [23]:
dataset = EvaluationDataset(
    goldens=[
        Golden(
            input="Que problema se concentra en los tickets de Initech?",
            multimodal=False,
        ),
        Golden(
            input="Porque la cuenta Acme declino en el Q2 de este año?",
            multimodal=False,
        ),
    ]
)

for golden in dataset.evals_iterator():
    app.invoke(
        {
            "question": golden.input,
            "answer": "",
        },
        config={"callbacks": [CallbackHandler()]},
    )

Output()

**************************************************

Answer Relevancy Verbose Logs

**************************************************

Statements:
[
    "En Q2-2026, la cuenta Acme enfrenta una caída o declinación explicada principalmente por churn alto y poco 
visible.",
    "Los ingresos se mantienen planos.",
    "El uso se desploma.",
    "Bajan los usuarios activos, los logins y la adopción.",
    "Hay riesgo de pérdida sin avisar porque la facturación no dispara alertas.",
    "La caída de uso sugiere desenganche real, no solo enojo.",
    "Hay fricción creciente alrededor del módulo de reportes y tableros.",
    "Existen bugs como filtros que no se guardan y exportaciones vacías.",
    "Hay más tickets de reportes personalizados.",
    "Esto coincide con un CSAT bajando ligeramente.",
    "El Director de TI, que era el champion, deja la empresa en mayo.",
    "Hay presión competitiva.",
    "Acme está evaluando activamente a StratusOne.",
    "StratusOne ofrece migración asistida y descuento agresivo.",
    "Esto puede acelerar la pérdida de adopción y elevar el riesgo hacia la renovación de Q3-2026."
] 
 
Verdicts:
[
    {
        "verdict": "yes",
        "reason": null
    },
    {
        "verdict": "yes",
        "reason": null
    },
    {
        "verdict": "yes",
        "reason": null
    },
    {
        "verdict": "yes",
        "reason": null
    },
    {
        "verdict": "yes",
        "reason": null
    },
    {
        "verdict": "yes",
        "reason": null
    },
    {
        "verdict": "yes",
        "reason": null
    },
    {
        "verdict": "yes",
        "reason": null
    },
    {
        "verdict": "yes",
        "reason": null
    },
    {
        "verdict": "yes",
        "reason": null
    },
    {
        "verdict": "yes",
        "reason": null
    },
    {
        "verdict": "yes",
        "reason": null
    },
    {
        "verdict": "yes",
        "reason": null
    },
    {
        "verdict": "yes",
        "reason": null
    },
    {
        "verdict": "yes",
        "reason": null
    }
]
 
Score: 1.0
Reason: The score is 1.00 because the response appears fully focused on explaining the Q2 decline for Acme using 
the requested types of causes and internal/financial evidence, with no irrelevant statements lowering its 
relevance. Strong job staying on topic and aligned with the input.

...

Truths (limit=None):
[
    "La cuenta corresponde a Initech.",
    "La nota de cuenta de Initech indica que el CSM es Diego Hernández.",
    "Initech pertenece al segmento Mid-Market.",
    "La renovación de la cuenta de Initech está indicada para Q1-2027.",
    "Existe un ticket con identificador TCK-00079 sobre cómo asignar roles a un nuevo usuario.",
    "El ticket TCK-00079 tiene fecha 2026-01-08.",
    "El ticket TCK-00079 está categorizado como Consulta / Cómo hacer.",
    "El ticket TCK-00079 tiene prioridad P3.",
    "En el ticket TCK-00079, Initech dice que está sumando gente al equipo.",
    "En el ticket TCK-00079, Initech quiere entender cómo asignar permisos por rol en Nimbus Core.",
    "En el ticket TCK-00079, Initech pregunta si hay una guía paso a paso.",
    "El contexto del trimestre Q2-2026 indica ingresos planos.",
    "El contexto del trimestre Q2-2026 indica fricción creciente alrededor del módulo de reportes y tableros.",
    "En Q2-2026 aumentaron los tickets de bugs relacionados con filtros que no se guardan y exportaciones vacías.",
    "En Q2-2026 aumentaron las solicitudes de reportes personalizados.",
    "En Q2-2026 la CSAT bajó ligeramente.",
    "Existe un ticket con identificador TCK-00001 sobre una duda del cobro de asientos adicionales.",
    "El ticket TCK-00001 tiene fecha 2026-01-20.",
    "El ticket TCK-00001 está categorizado como Facturación.",
    "El ticket TCK-00001 tiene prioridad P3.",
    "En el ticket TCK-00001, Initech reporta un incremento en el monto facturado.",
    "En el ticket TCK-00001, Initech solicita el desglose de asientos y el prorrateo aplicado ese mes.",
    "Existe un ticket con identificador TCK-00004 sobre una duda del cobro de asientos adicionales.",
    "El ticket TCK-00004 tiene fecha 2026-01-07.",
    "El ticket TCK-00004 está categorizado como Facturación.",
    "El ticket TCK-00004 tiene prioridad P2.",
    "En el ticket TCK-00004, Initech reporta un incremento en el monto facturado.",
    "En el ticket TCK-00004, Initech solicita el desglose de asientos y el prorrateo aplicado ese mes.",
    "Existe un ticket con identificador TCK-00041 sobre una duda del cobro de asientos adicionales.",
    "El ticket TCK-00041 tiene fecha 2026-06-21.",
    "El ticket TCK-00041 está categorizado como Facturación.",
    "El ticket TCK-00041 tiene prioridad P2.",
    "En el ticket TCK-00041, Initech reporta un incremento en el monto facturado.",
    "En el ticket TCK-00041, Initech solicita el desglose de asientos y el prorrateo aplicado ese mes.",
    "Existe un ticket con identificador TCK-00022 titulado 'Caída durante horario crítico'.",
    "El ticket TCK-00022 tiene fecha 2026-04-21.",
    "El ticket TCK-00022 está categorizado como Caída del servicio.",
    "El ticket TCK-00022 tiene prioridad P1.",
    "En el ticket TCK-00022, Initech indica que el servicio se cayó durante su cierre de turno.",
    "En el ticket TCK-00022, Initech afirma que la falta de disponibilidad les costó varias horas de trabajo.",
    "En el ticket TCK-00022, Initech solicita un reporte post-incidente formal.",
    "Existe un ticket con identificador TCK-00025 titulado 'Caída durante horario crítico'.",
    "El ticket TCK-00025 tiene fecha 2026-05-29.",
    "El ticket TCK-00025 está categorizado como Caída del servicio.",
    "El ticket TCK-00025 tiene prioridad P1.",
    "En el ticket TCK-00025, Initech indica que el servicio se cayó durante su cierre de turno.",
    "En el ticket TCK-00025, Initech afirma que la falta de disponibilidad les costó varias horas de trabajo.",
    "En el ticket TCK-00025, Initech solicita un reporte post-incidente formal.",
    "Existe un ticket con identificador TCK-00042 titulado 'Caída durante horario crítico'.",
    "El ticket TCK-00042 tiene fecha 2026-06-02.",
    "El ticket TCK-00042 está categorizado como Caída del servicio.",
    "El ticket TCK-00042 tiene prioridad P1.",
    "En el ticket TCK-00042, Initech indica que el servicio se cayó durante su ci

======================================================================

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 0 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_1 (Passed 0 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ answer_question                                                                                             │
│  ├──   Input:            ¿Por qué la cuenta (o cliente) “Acme” registró una disminución o declinación en el     │
│  │                       Q2 de este año? Analiza causas posibles (ingresos/ventas, churn, facturación,          │
│  │                       cambios de contrato, descuentos, problemas operativos o de servicio) y cita datos o    │
│  │                       fuentes internas/financieras disponibles para ese trimestre.                           │
│  │     Actual Output:    En Q2-2026, la cuenta **Acme** enfrenta una **caída/declinación explicada              │
│  │                       principalmente por churn “ALTO y poco visible”**: los **ingresos se mantienen          │
│  │                       planos**, pero el **uso se desploma** (bajan usuarios activos, logins y adopción) y    │
│  │                       hay **riesgo de pérdida “sin avisar”** porque **la facturación no dispara              │
│  │                       alertas**; además, la caída de uso sugiere **desenganche real**, no solo enojo.        │
│  │                       Como factores internos/operativos, el contexto del trimestre también menciona          │
│  │                       **fricción creciente** alrededor del módulo de **reportes y tableros** (bugs como      │
│  │                       filtros que no se guardan/exportaciones vacías y más tickets de reportes               │
│  │                       personalizados), y eso coincide con **CSAT bajando ligeramente**. Por último, hay      │
│  │                       cambios clave de negocio: **salida del champion** (Director de TI deja la empresa      │
│  │                       en mayo) y **presión competitiva** (Acme evaluando activamente a **StratusOne** con    │
│  │                       **migración asistida** y **descuento agresivo**), lo que puede acelerar la pérdida     │
│  │                       de adopción y elevar el riesgo hacia la renovación de **Q3-2026**.                     │
│  └── Metrics                                                                                                    │
│       Status  ┃ Metric            ┃ Score  ┃ Threshold  ┃ Reason                                                │
│      ━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        PASS   │ Answer Relevancy  │ 1.00   │ 0.50       │ The score is 1.00 because the response appears ...    │
│        PASS   │ Faithfulness      │ 1.00   │ 0.50       │ The score is 1.00 because there are no contradi...    │
│                                                           

⚠ WARNING: No hyperparameters logged.
» ]8;id=12588570;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 17.31s | token cost: None)
» Test Results (2 total tests):
   » Pass Rate: 100.0% | Passed: 2 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

---

## Calibración

El tema de calibración es importante porque las respuestas de un LLM pueden sufrir de sesgo de confianza y posición, verbosidad, auto-preferencia, formato y se mitigan con las estrategias: intercambio de orden, normalización de longitud, ensembles de familias distintas y forzar el razonamiento antes del veredicto.


### Calibración con Probe

Este tipo de técnica suele abordar el problema de una forma diferente puesto que mide la correctitud de la respuesta con base en el flujo de tokens del Modelo en capas intermedias.

¿Por qué en capas intermedias?

Los Modelos en sus capas iniciales suelen ser de bajo nivel, es decir no tienen una representación rica. Las capas más profundas están muy especializadas para la predicción del siguiente token, en contraste las capas intermedias retienen la mayor representación semántica de toda la red.

Métricas

Para evaluar la precisión del algoritmo se toman las métricas Kuiper y Expected Calibration Error las cuales miden el desajuste acumulado de calibración y la cuantificación entre la confianza predicha y la precisión real respectivamente.

Realidad

En lo que va del 2026 las tecnicas de evaluación usando fine-tunning o calibración de la respuesta han sido reemplazadas por evaluaciones usando modelos de frontera

---

# Muchas Gracias!

## Github: github.com/jmanuelc87
## Linkedin: linkedin.com/in/jmanuelc87